<a href="https://colab.research.google.com/github/zhangyr0212-ui/hopenhay_claude/blob/main/gender_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from collections import namedtuple

jax.config.update("jax_enable_x64", True)

# =========================================================
# 1. PARAMETERS / GRIDS
# =========================================================

Parameters = namedtuple("Parameters",
    ("β", "θ", "c", "w",
     "m_a", "m_b", "σ",
     "m_c", "m_d", "δ"))

Grids = namedtuple("Grids",
    ("φ_grid_1", "φ_grid_2",
     "E_draws_1", "E_draws_2",
     "A_draws_1", "A_draws_2"))

Model = namedtuple("Model", ("parameters", "grids"))


def create_model(
    β=0.95, θ=0.6, c=0.01, w=1.0,
    m_a=-0.1, m_b=-0.1, σ=0.2,
    m_c=-0.2, m_d=-0.1, δ=0.3,
    φ_grid_max=5, φ_grid_size=100,
    E_draw_size=200, A_draw_size=200,
    seed=1234
):
    assert m_a + σ**2 / (2 * (1 - θ)) < 0, "Stability condition fails"
    assert m_b + σ**2 / (2 * (1 - θ)) < 0, "Stability condition fails"
    φ_grid_1 = jnp.linspace(0, φ_grid_max, φ_grid_size)
    φ_grid_2 = jnp.linspace(0, φ_grid_max, φ_grid_size)

    k1, k2, k3, k4 = jax.random.split(jax.random.PRNGKey(seed), 4)

    A_draws_1 = jnp.exp(m_a + σ * jax.random.normal(k1, (A_draw_size,)))
    A_draws_2 = jnp.exp(m_b + σ * jax.random.normal(k2, (A_draw_size,)))

    E_draws_1 = jnp.exp(m_c + δ * jax.random.normal(k3, (E_draw_size,)))
    E_draws_2 = jnp.exp(m_d + δ * jax.random.normal(k4, (E_draw_size,)))

    parameters = Parameters(β, θ, c, w, m_a, m_b, σ, m_c, m_d, δ)
    grids = Grids(φ_grid_1, φ_grid_2,
                  E_draws_1, E_draws_2,
                  A_draws_1, A_draws_2)
    return Model(parameters, grids)


# =========================================================
# 2. PROFIT / OUTPUT
# =========================================================
@jax.jit
def π(φ, p, parameters):
    β, θ, c, w, *_ = parameters
    return (1 - θ) * (p * φ)**(1/(1-θ)) * (θ/w)**(θ/(1-θ)) - c

@jax.jit
def q(φ, p, parameters):
    β, θ, c, w, *_ = parameters
    return φ**(1/(1-θ)) * (p * θ/w)**(θ/(1-θ))

# =========================================================
# 5. UPDATE CROSS SECTION
# =========================================================

def update_cross_section(φ_bar, φ_vec, key, parameters, num_firms, type_id):
    # Unpack
    β, θ, c, w, m_a, m_b, σ, m_c, m_d, δ = parameters


    Z = jax.random.normal(key, (2, num_firms))


    if type_id == 1:
        incumbent_draws = φ_vec * jnp.exp(m_a + σ * Z[0, :])
        new_firm_draws  = jnp.exp(m_c + δ * Z[1, :])
    else:
        incumbent_draws = φ_vec * jnp.exp(m_b + σ * Z[0, :])
        new_firm_draws  = jnp.exp(m_d + δ * Z[1, :])


    return jnp.where(φ_vec >= φ_bar, incumbent_draws, new_firm_draws)

# =========================================================
# 6. SIMULATION
# =========================================================

def simulate_firms(φ_bar, parameters, grids, type_id,
                   sim_length=200, num_firms=1_000_000, seed=12):

    # Set initial conditions to the threshold value
    φ_vec = jnp.ones((num_firms,)) * φ_bar
    key = jax.random.PRNGKey(seed)

    # Iterate forward in time
    for t in range(sim_length):
        key, subkey = jax.random.split(key)
        φ_vec = update_cross_section(φ_bar, φ_vec, subkey,
                                     parameters, num_firms, type_id)

    return φ_vec


# =========================================================
# 3. EXPECTATION OPERATOR (3-LAYER STRUCTURE)
# =========================================================

def _compute_exp_value_at_phi(v1, v2, φ, grids, type_id):
    """
    Compute E[v(A φ) | φ] using interpolation and Monte Carlo.
    """
    φ_grid_1, φ_grid_2, E1, E2, A1, A2 = grids


    if type_id == 1:
        φ_grid = φ_grid_1
        A_draws = A1
        v = v1
    else:
        φ_grid = φ_grid_2
        A_draws = A2
        v = v2

    Aφ = A_draws * φ
    vAφ = jnp.interp(Aφ, φ_grid, v)

    return jnp.mean(vAφ)


compute_exp_value_at_phi = jax.vmap(
    _compute_exp_value_at_phi,
    in_axes=(None, None, 0, None, None)
)


def compute_exp_value(v1, v2, grids, type_id):
    """
    Compute E[v(A φ) | φ] for all φ.
    """
    φ_grid_1, φ_grid_2, *_ = grids

    if type_id == 1:
        φ_grid = φ_grid_1
    else:
        φ_grid = φ_grid_2

    return compute_exp_value_at_phi(v1, v2, φ_grid, grids, type_id)


# =========================================================
# 4. BELLMAN OPERATOR
# =========================================================


def T(v1, v2, p, parameters, grids, type_id):

    β, θ, c, w, *_ = parameters


    EvAφ = compute_exp_value(v1, v2, grids, type_id)

    φ_grid = grids[0] if type_id == 1 else grids[1]

    return π(φ_grid, p, parameters) + β * jnp.maximum(0.0, EvAφ)

# =========================================================
# 8. EXIT THRESHOLD
# =========================================================


def get_threshold(v1, v2, grids, type_id):


    φ_grid = grids[0] if type_id == 1 else grids[1]


    EvAφ = compute_exp_value(v1, v2, grids, type_id)


    i = jnp.searchsorted(EvAφ, 0.0)

    return φ_grid[i]

# =========================================================
# 7. VALUE FUNCTION ITERATION
# =========================================================


def vfi(p, v1_init, v2_init, parameters, grids,
        tol=1e-6, max_iter=10_000):
    φ_grid_1, φ_grid_2, E1, E2, A1, A2 = grids
    def cond(state):
        i, v1, v2, err = state
        return jnp.logical_and (i < max_iter, err > tol)

    def body(state):
        i, v1, v2, err = state

        new_v1 = T(v1, v2, p, parameters, grids, 1)
        new_v2 = T(v1, v2, p, parameters, grids, 2)

        err1 = jnp.max(jnp.abs( v1 - new_v1))
        err2 = jnp.max(jnp.abs( v2 - new_v2 ))
        err = jnp.maximum(err1, err2)
        i += 1
        return i, new_v1, new_v2, err

    init_state = 0, v1_init, v2_init, tol + 1
    state = jax.lax.while_loop(cond, body, init_state)
    i, v1, v2, err = state
    return v1, v2




# =========================================================
# 9. MARKET CLEARING: q1 + q2 = 1/p
# =========================================================

def market_clearing(p, model,
                    v1_init, v2_init,
                    sim_length=200,
                    num_firms=200_000,
                    seed=123):

    parameters, grids = model

    # =====================================================
    # 1. Solve value functions
    # =====================================================
    v1, v2 = vfi(p, v1_init, v2_init, parameters, grids)

    # =====================================================
    # 2. Get exit thresholds
    # =====================================================
    φ_bar_1 = get_threshold(v1, v2, grids, 1)
    φ_bar_2 = get_threshold(v1, v2, grids, 2)

    # =====================================================
    # 3. Simulate stationary distributions
    # =====================================================
    φ_dist_1 = simulate_firms(φ_bar_1, parameters, grids,
                             type_id=1,
                             sim_length=sim_length,
                             num_firms=num_firms,
                             seed=seed)

    φ_dist_2 = simulate_firms(φ_bar_2, parameters, grids,
                             type_id=2,
                             sim_length=sim_length,
                             num_firms=num_firms,
                             seed=seed+1)

    # =====================================================
    # 4. Compute outputs
    # =====================================================
    q1 = q(φ_dist_1, p, parameters)
    q2 = q(φ_dist_2, p, parameters)

    # =====================================================
    # 5. Aggregate supply
    # =====================================================
    Y = jnp.mean(q1) + jnp.mean(q2)

    return Y
def excess_demand(p, model, v1_init, v2_init):
    Y = market_clearing(p, model, v1_init, v2_init)
    return Y - 1.0 / p
def compute_p_star(model, v1_init, v2_init,
                   p_min=0.1, p_max=10.0, tol=1e-5):

    lower, upper = p_min, p_max

    while upper - lower > tol:
        mid = 0.5 * (upper + lower)

        excess = excess_demand(mid, model, v1_init, v2_init)

        if excess > 0:   # supply > demand → price too high
            upper = mid
        else:            # supply < demand → price too low
            lower = mid

    p_star = 0.5 * (upper + lower)

    return p_star


# =========================================================
# 10. EQUILIBRIUM
# =========================================================

def compute_equilibrium(model):

    parameters, grids = model

    # =====================================================
    # 0. Initial value functions
    # =====================================================
    v1_init = jnp.zeros_like(grids.φ_grid_1)
    v2_init = jnp.zeros_like(grids.φ_grid_2)

    # =====================================================
    # 1. Solve for equilibrium price
    # =====================================================
    p_star = compute_p_star(model, v1_init, v2_init)

    # =====================================================
    # 2. Solve value functions at p*
    # =====================================================
    v1, v2 = vfi(p_star, v1_init, v2_init, parameters, grids)

    # =====================================================
    # 3. Exit thresholds
    # =====================================================
    φ_bar_1 = get_threshold(v1, v2, grids, 1)
    φ_bar_2 = get_threshold(v1, v2, grids, 2)

    # =====================================================
    # 4. (Optional) simulate distributions
    # =====================================================
    φ_dist_1 = simulate_firms(φ_bar_1, parameters, grids, 1)
    φ_dist_2 = simulate_firms(φ_bar_2, parameters, grids, 2)

    # =====================================================
    # 5. Compute outputs
    # =====================================================
    q1 = q(φ_dist_1, p_star, parameters)
    q2 = q(φ_dist_2, p_star, parameters)

    Y = jnp.mean(q1) + jnp.mean(q2)

    return {
        "p_star": p_star,
        "φ_bar_1": φ_bar_1,
        "φ_bar_2": φ_bar_2,
        "Y": Y,
        "v1": v1,
        "v2": v2
    }


# =========================================================
# 11. ENTRY VALUE
# =========================================================
def compute_entry_value(p, v1_init, v2_init, model, type_id):

    parameters, grids = model

    # =====================================================
    # 1. Solve value functions
    # =====================================================
    v1, v2 = vfi(p, v1_init, v2_init, parameters, grids)

    # =====================================================
    # 2. Select type-specific objects
    # =====================================================
    if type_id == 1:
        φ_grid = grids.φ_grid_1
        E_draws = grids.E_draws_1
        v = v1
    else:
        φ_grid = grids.φ_grid_2
        E_draws = grids.E_draws_2
        v = v2

    # =====================================================
    # 3. Compute expected entry value
    # =====================================================
    v_φ = jnp.interp(E_draws, φ_grid, v)
    Ev_φ = jnp.mean(v_φ)

    return Ev_φ



# =========================================================
# RUN
# =========================================================

# =========================================================
# RUN
# =========================================================

def run_model():

    # =====================================================
    # 1. Create model
    # =====================================================
    model = create_model()
    parameters, grids = model

    # =====================================================
    # 2. Initial value functions
    # =====================================================
    v1_init = jnp.zeros_like(grids.φ_grid_1)
    v2_init = jnp.zeros_like(grids.φ_grid_2)

    # =====================================================
    # 3. Compute equilibrium
    # =====================================================
    result = compute_equilibrium(model)

    p_star = result["p_star"]
    φ_bar_1 = result["φ_bar_1"]
    φ_bar_2 = result["φ_bar_2"]

    # =====================================================
    # 4. Compute entry values (c1, c2)
    # =====================================================
    c1 = compute_entry_value(p_star, v1_init, v2_init, model, type_id=1)
    c2 = compute_entry_value(p_star, v1_init, v2_init, model, type_id=2)

    # =====================================================
    # 5. Print results
    # =====================================================
    print("===================================")
    print(f"p*       = {p_star:.6f}")
    print(f"φ_bar_1  = {φ_bar_1:.6f}")
    print(f"φ_bar_2  = {φ_bar_2:.6f}")
    print(f"c1 (EV1) = {c1:.6f}")
    print(f"c2 (EV2) = {c2:.6f}")
    print("===================================")


# =========================================================
# EXECUTE
# =========================================================

run_model()